# 02 — Custom CNN Training (from scratch)

Trains the 4-block custom CNN defined in `src/models.py` on the identity subset
prepared in `01_data_preparation.ipynb`. Saves the best checkpoint, training curves,
and history to `logs/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
from torch.utils.data import DataLoader

from dataset import CelebASubset
from models import CustomCNN
from utils import set_seed, get_device, train_model

set_seed(42)
device = get_device()
print("device:", device)


In [ ]:
BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-3

train_ds = CelebASubset("train")
val_ds = CelebASubset("val")
num_classes = train_ds.num_classes
print("num_classes:", num_classes, "| train:", len(train_ds), "| val:", len(val_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
model = CustomCNN(num_classes=num_classes)
print(f"CustomCNN parameters: {model.num_params():,}")


In [ ]:
history, train_time_sec, best_val_acc = train_model(
    model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, device=device, run_name="custom_cnn",
)
print(f"\nBest val accuracy: {best_val_acc:.4f}  |  Training time: {train_time_sec:.1f}s")


**Notes for CPU-only runs (Explorer):** if a compute node kernel dies with no
traceback while this cell runs, that node likely lacks AVX2 — request a different
node/partition, or drop `BATCH_SIZE`/`IMAGE_SIZE` further. 25 epochs on ~5 classes at
128x128 should complete in well under an hour on a modern CPU; reduce `EPOCHS` if
iterating.